<a href="https://colab.research.google.com/github/alheliou/Bias_mitigation/blob/main/UPP26/TD4_pre_post.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TD 4: Mitigation des biais avec des méthodes de pré-processing et de post-processing



## Installation of the environnement

We highly recommend you to follow these steps, it will allow every student to work in an environment as similar as possible to the one used during testing.

### Colab Settings ---- for Colab Users ONLY
  The next cell of code are to execute only once per colab environment


#### Python env creation (Colab only)

        ```
        ! python -m pip install numpy fairlearn plotly nbformat ipykernel aif360["inFairness"] aif360['AdversarialDebiasing'] causal-learn BlackBoxAuditing cvxpy dice-ml lime shapkit
        ```
#### 2. Download MEPS dataset (for part2) it can take several minutes (Colab only)

        ```
        ! Rscript /usr/local/lib/python3.12/dist-packages/aif360/data/raw/meps/generate_data.R
        ! mv h181.csv /usr/local/lib/python3.12/dist-packages/aif360/data/raw/meps/
        ! mv h192.csv /usr/local/lib/python3.12/dist-packages/aif360/data/raw/meps/
        ```

### Local Settings ---- for installation on local computer ONLY

If you arleady have an env from TD2 or TD3, you can simply reuse it.


#### 1. Uv installation (local only, no need to redo if already done)


        https://docs.astral.sh/uv/getting-started/installation/


        `curl -LsSf https://astral.sh/uv/install.sh | sh`

        Python version 3.12 installation (highly recommended)
        `uv python install 3.12`

#### 2. R installation *NEW* (local only)

        In the command `Rscript` says 'command not found'

        `sudo apt install r-base-core`

#### 3. Python env creation (local only, no need to redo if already done)

        ```
        mkdir TD_bias_mitigation
        cd TD_bias_mitigation
        uv python pin 3.12
        uv init
        uv venv
        uv add numpy fairlearn plotly nbformat ipykernel aif360["inFairness"] aif360['AdversarialDebiasing'] causal-learn BlackBoxAuditing cvxpy dice-ml lime shapkit
        uv add pandas==2.2.2
        ```

#### 4. Download MEPS dataset, it can take several minutes *NEW* (local only)

        ```
        cd TD_bias_mitigation/.venv/lib/python3.12/site-packages/aif360/data/raw/meps/
        Rscript generate_data.R
        ```

In [ ]:
# To execute only in Colab
! python -m pip install numpy fairlearn plotly nbformat ipykernel aif360["inFairness"] aif360['AdversarialDebiasing'] causal-learn BlackBoxAuditing cvxpy dice-ml lime shapkit

In [ ]:
# To execute only in Colab
! Rscript /usr/local/lib/python3.12/dist-packages/aif360/data/raw/meps/generate_data.R
! mv h181.csv /usr/local/lib/python3.12/dist-packages/aif360/data/raw/meps/
! mv h192.csv /usr/local/lib/python3.12/dist-packages/aif360/data/raw/meps/

## 1.Manipulate the dataset

In [1]:
# imports
import numpy as np
import pandas as pd
import plotly.express as px
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", append=True, category=UserWarning)
# Datasets
from aif360.datasets import MEPSDataset19
from aif360.explainers import MetricTextExplainer

# Fairness metrics
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.metrics import ClassificationMetric
from sklearn.metrics import accuracy_score, balanced_accuracy_score


MEPSDataset19_data = MEPSDataset19()
(dataset_orig_panel19_train, dataset_orig_panel19_val, dataset_orig_panel19_test) = (
    MEPSDataset19().split([0.5, 0.8], shuffle=True)
)

2026-02-06 09:30:43.853643: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-06 09:30:44.712088: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-06 09:30:47.704884: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
len(dataset_orig_panel19_train.instance_weights), len(
    dataset_orig_panel19_val.instance_weights
), len(dataset_orig_panel19_test.instance_weights)

(7915, 4749, 3166)

In [3]:
instance_weights = MEPSDataset19_data.instance_weights
instance_weights

array([21854.981705, 18169.604822, 17191.832515, ...,  3896.116219,
        4883.851005,  6630.588948], shape=(15830,))

In [4]:
f"Taille du dataset {len(instance_weights)}, poids total du dataset {instance_weights.sum()}."

'Taille du dataset 15830, poids total du dataset 141367240.546316.'

Conversion en dataframe

In [5]:
def get_df(MepsDataset):
    data = MepsDataset.convert_to_dataframe()
    # data_train est un tuple, avec le data_frame et un dictionnaire avec toutes les infos (poids, attributs sensibles etc)
    df = data[0]
    df["WEIGHT"] = data[1]["instance_weights"]
    return df


df = get_df(MEPSDataset19_data)

Nous réalisons maintenant l'opération inverse (qui sera indispensable pour le projet). Créer un objet de la classe StandardDataset de AIF360 à partir du dataframe.

Pour le projet cela vous permettre d'utiliser les méthode déjà implémentées dans AIF360 sur votre jeu de données.

Ici cela n'a aucun intéret car le dataframe vien d'un StandardDataset, nous vous fournissons le code. Mais cela vaut le coup de le lire attentivement et de poser des questions si besoin.




In [6]:
import os
from aif360.datasets import StandardDataset
import pandas as pd

# Get categorical column from one hot encoding (specitic to MEPSdataset)
# Here we create a dictionnary that links each categorical column name
# to the list of corresponding one hot encoded columns
categorical_columns_dic = {}
for col in df.columns:
    col_split = col.split("=")
    if len(col_split) > 1:
        cat_col = col_split[0]
        if not (cat_col in categorical_columns_dic.keys()):
            categorical_columns_dic[cat_col] = []
        categorical_columns_dic[cat_col].append(col)
categorical_features = categorical_columns_dic.keys()

In [7]:
# Now we recreate the categorical column value from the one hot encoded
print(df.shape)


def categorical_transform(df, onehotencoded, cat_col):
    if len(onehotencoded) > 1:
        return df[onehotencoded].apply(
            lambda x: onehotencoded[np.argmax(x)][len(cat_col) + 1 :], axis=1
        )
    else:
        return df[onehotencoded]


# Reverse the categorical one hot encoded
for cat_col, onehotencoded in categorical_columns_dic.items():
    df[cat_col] = categorical_transform(df, onehotencoded, cat_col)
    df.drop(columns=onehotencoded, inplace=True)

df.shape

(15830, 140)


(15830, 43)

In [8]:
MyDataset = StandardDataset(
    df=df,
    label_name="UTILIZATION",
    favorable_classes=[1],
    protected_attribute_names=["RACE"],
    privileged_classes=[[1]],
    instance_weights_name="WEIGHT",
    categorical_features=categorical_features,
    features_to_keep=[],
    features_to_drop=[],
    na_values=["?", "Unknown/Invalid"],
    custom_preprocessing=None,
    metadata=None,
)

In [9]:
# We check the dataset has the same metrics :D
# Attention étonnanement le positive label 'favorable_classes' est par défaut 1 (cela est un peu bizarre pour ce dataset)
print(
    BinaryLabelDatasetMetric(
        MEPSDataset19_data,
        unprivileged_groups=[{"RACE": 0}],
        privileged_groups=[{"RACE": 1}],
    ).disparate_impact(),
    BinaryLabelDatasetMetric(
        MEPSDataset19_data,
        unprivileged_groups=[{"RACE": 0}],
        privileged_groups=[{"RACE": 1}],
    ).base_rate(),
)
print(
    BinaryLabelDatasetMetric(
        MyDataset, unprivileged_groups=[{"RACE": 0}], privileged_groups=[{"RACE": 1}]
    ).disparate_impact(),
    BinaryLabelDatasetMetric(
        MyDataset, unprivileged_groups=[{"RACE": 0}], privileged_groups=[{"RACE": 1}]
    ).base_rate(),
)

0.49826823461176517 0.21507139363038463
0.49826823461176517 0.21507139363038463


In [10]:
from aif360.sklearn.metrics import disparate_impact_ratio, base_rate

dir = disparate_impact_ratio(
    y_true=df.UTILIZATION, prot_attr=df.RACE, pos_label=1, sample_weight=df.WEIGHT
)
br = base_rate(y_true=df.UTILIZATION, pos_label=1, sample_weight=df.WEIGHT)
dir, br

(0.4982682346117653, np.float64(0.21507139363038463))

## 2. Appliquer les méthodes de pré-processing disponibles dans AIF360

In [11]:
sens_ind = 0
sens_attr = dataset_orig_panel19_train.protected_attribute_names[sens_ind]
unprivileged_groups = [
    {sens_attr: v}
    for v in dataset_orig_panel19_train.unprivileged_protected_attributes[sens_ind]
]
privileged_groups = [
    {sens_attr: v}
    for v in dataset_orig_panel19_train.privileged_protected_attributes[sens_ind]
]
sens_attr, unprivileged_groups, privileged_groups

('RACE', [{'RACE': np.float64(0.0)}], [{'RACE': np.float64(1.0)}])

### 2.1 Quesiton: Apprendre une regression logistique qui prédit l'UTILIZATION

Attention nous avons enlever le preprocessing sur le dataframe, il faut cette fois utiliser l'API d'AIF360
https://aif360.readthedocs.io/en/latest/modules/generated/aif360.datasets.StructuredDataset.html

pour retrouver les features (X), les labels (y) et les poids de chaque instance du dataset

In [12]:
from sklearn.linear_model import LogisticRegression

X_train = MyDataset.features
y_train = MyDataset.labels.ravel()
w_train = MyDataset.instance_weights

clf = LogisticRegression(solver="liblinear", max_iter=1000)
clf.fit(X_train, y_train, sample_weight=w_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

### 2.2 Question: Calcul des métriques de fairness

Calculer les métriques du dataset de validation seul.

Calculer les métriques basées sur les prédictions et la vérité du dataset de validation.

En comparaison calculer les métriques basées sur des prédictions aléatoires et la vérité du dataset de validation.

In [13]:
val_metric = BinaryLabelDatasetMetric(
    dataset_orig_panel19_val,
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups,
)

print("Base rate:", val_metric.base_rate())
print("Disparate impact:", val_metric.disparate_impact())
print("Statistical parity difference:", val_metric.statistical_parity_difference())
print("Mean difference:", val_metric.mean_difference())

Base rate: 0.21063996784930955
Disparate impact: 0.46875940682903
Statistical parity difference: -0.14220746363970482
Mean difference: -0.14220746363970482


In [14]:
print("Metrics based on predictions and truth on validation dataset")

# Train
X_train = dataset_orig_panel19_train.features
y_train = dataset_orig_panel19_train.labels.ravel()
w_train = dataset_orig_panel19_train.instance_weights

clf = LogisticRegression(solver="liblinear", max_iter=1000)
clf.fit(X_train, y_train, sample_weight=w_train)

# Predict on val
X_val = dataset_orig_panel19_val.features
y_val_pred = clf.predict(X_val)

# Build predicted dataset
dataset_val_pred = dataset_orig_panel19_val.copy(deepcopy=True)
dataset_val_pred.labels = y_val_pred.reshape(-1, 1)

Metrics based on predictions and truth on validation dataset


In [16]:
print("Metrics based on random predictions and truth on validation dataset")
n = dataset_orig_panel19_val.labels.shape[0]

val_metric = BinaryLabelDatasetMetric(
    dataset_orig_panel19_val,
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups,
)
p = val_metric.base_rate()  # proba de prédire 1

y_rand_cal = (np.random.rand(n) < p).astype(int)

dataset_val_rand_cal = dataset_orig_panel19_val.copy(deepcopy=True)
dataset_val_rand_cal.labels = y_rand_cal.reshape(-1, 1)

rand_cal_metric = ClassificationMetric(
    dataset_orig_panel19_val,
    dataset_val_rand_cal,
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups,
)

print("\nRandom (calibrated) accuracy:", rand_cal_metric.accuracy())
print("Random (calibrated) statistical parity difference:", rand_cal_metric.statistical_parity_difference())
print("Random (calibrated) disparate impact:", rand_cal_metric.disparate_impact())
print("Random (calibrated) equal opportunity difference:", rand_cal_metric.equal_opportunity_difference())
print("Random (calibrated) average odds difference:", rand_cal_metric.average_odds_difference())


Metrics based on random predictions and truth on validation dataset

Random (calibrated) accuracy: 0.6684917901005023
Random (calibrated) statistical parity difference: 0.01218605603771794
Random (calibrated) disparate impact: 1.0599606234499628
Random (calibrated) equal opportunity difference: 0.050887677724291286
Random (calibrated) average odds difference: 0.027841260561322953


### 2.2 Repondération
#### 2.2.1. Question : Trouver dans l'API quels objets/fonctions sont à utiliser pour faire de repondération et les appliquer sur le dataset d'apprentissage

In [19]:
from aif360.algorithms.preprocessing import Reweighing
# We use the Reweighing algorithm to preprocess the training dataset and try to mitigate bias before training the model
RW = Reweighing(
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)

dataset_train_rw = RW.fit_transform(dataset_orig_panel19_train)

#### 2.2.2. Question: Apprendre une regression logistique sur les données pondérées et calculer les métriques de fairness sur l'échantillon de validation

Comme vu en cours le Reweighting ne modifie que la pondération du dataset, les features et label restent inchangés.

In [20]:
from aif360.algorithms.preprocessing import Reweighing
from aif360.metrics import ClassificationMetric
from sklearn.linear_model import LogisticRegression
import numpy as np

# On fait un Reweighing sur le TRAIN seulement (ne change que instance_weights)
RW = Reweighing(
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)

dataset_train_rw = RW.fit_transform(dataset_orig_panel19_train)

# Entraîner une régression logistique avec les poids repondérés
X_train = dataset_train_rw.features
y_train = dataset_train_rw.labels.ravel()
w_train = dataset_train_rw.instance_weights

clf_rw = LogisticRegression(solver="liblinear", max_iter=1000)
clf_rw.fit(X_train, y_train, sample_weight=w_train)

# Prédire sur le dataset de validation (NON repondéré)
X_val = dataset_orig_panel19_val.features
y_val_pred = clf_rw.predict(X_val)

dataset_val_pred_rw = dataset_orig_panel19_val.copy(deepcopy=True)
dataset_val_pred_rw.labels = y_val_pred.reshape(-1, 1)

# Calcul des métriques fairness (pred vs truth) sur la validation
metric_rw = ClassificationMetric(
    dataset_orig_panel19_val,      # vérité
    dataset_val_pred_rw,           # prédictions
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)

print("Accuracy:", metric_rw.accuracy())
print("Statistical parity difference (SPD):", metric_rw.statistical_parity_difference())
print("Disparate impact (DI):", metric_rw.disparate_impact())
print("Equal opportunity difference (EOD):", metric_rw.equal_opportunity_difference())
print("Average odds difference (AOD):", metric_rw.average_odds_difference())


Accuracy: 0.8320903748358845
Statistical parity difference (SPD): -0.03744543614019985
Disparate impact (DI): 0.7170999283040798
Equal opportunity difference (EOD): 0.0752921051490772
Average odds difference (AOD): 0.036327751751205214


### 2.3. Disparate Impact Remover
#### 2.3.1. Question : Trouver dans l'API quels objets/fonctions sont à utiliser pour faire une approache de disparate impact remover et les appliquer.


In [23]:
from aif360.algorithms.preprocessing import DisparateImpactRemover

DIR = DisparateImpactRemover(
    repair_level=1.0,
    sensitive_attribute=sens_attr  # "RACE"
)

dataset_train_dir = DIR.fit_transform(dataset_orig_panel19_train)
dataset_val_dir   = DIR.fit_transform(dataset_orig_panel19_val)



#### 2.3.2. Question: Apprendre une regression logistique sur les données transformées en retirant l'attribut sensible et calculer les métriques de fairness sur l'échantillon de validation

In [25]:
def drop_feature(dataset, feature_name: str):
    """
    Retourne une copie du dataset AIF360 en retirant la feature 'feature_name'
    de dataset.features + dataset.feature_names.
    (On ne touche pas aux protected_attributes.)
    """
    if feature_name not in dataset.feature_names:
        # Rien à enlever : on renvoie une copie
        return dataset.copy(deepcopy=True)

    keep_idx = [i for i, f in enumerate(dataset.feature_names) if f != feature_name]

    ds2 = dataset.copy(deepcopy=True)
    ds2.features = ds2.features[:, keep_idx]
    ds2.feature_names = [dataset.feature_names[i] for i in keep_idx]
    return ds2


In [26]:
print("2.3.2 DIR + remove sensitive feature + Logistic Regression + fairness metrics on validation")

from aif360.algorithms.preprocessing import DisparateImpactRemover
from aif360.metrics import ClassificationMetric
from sklearn.linear_model import LogisticRegression

# 1) DIR (dans ta version, pas de transform() -> fit_transform sur chaque split)
DIR = DisparateImpactRemover(repair_level=1.0, sensitive_attribute=sens_attr)

dataset_train_dir = DIR.fit_transform(dataset_orig_panel19_train)
dataset_val_dir   = DIR.fit_transform(dataset_orig_panel19_val)

# 2) Retirer l'attribut sensible des FEATURES
dataset_train_dir_wo = drop_feature(dataset_train_dir, sens_attr)
dataset_val_dir_wo   = drop_feature(dataset_val_dir, sens_attr)

# 3) Entraîner la régression logistique sur train transformé + sans attribut sensible
clf_dir = LogisticRegression(solver="liblinear", max_iter=1000)
clf_dir.fit(
    dataset_train_dir_wo.features,
    dataset_train_dir_wo.labels.ravel(),
    sample_weight=dataset_train_dir_wo.instance_weights
)

# 4) Prédire sur val transformé + sans attribut sensible
y_val_pred = clf_dir.predict(dataset_val_dir_wo.features)

dataset_val_pred = dataset_orig_panel19_val.copy(deepcopy=True)
dataset_val_pred.labels = y_val_pred.reshape(-1, 1)

# 5) Métriques fairness sur validation (truth vs pred)
metric = ClassificationMetric(
    dataset_orig_panel19_val,
    dataset_val_pred,
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)

print("Accuracy:", metric.accuracy())
print("SPD:", metric.statistical_parity_difference())
print("DI:", metric.disparate_impact())
print("EOD:", metric.equal_opportunity_difference())
print("AOD:", metric.average_odds_difference())


2.3.2 DIR + remove sensitive feature + Logistic Regression + fairness metrics on validation
Accuracy: 0.8314360525123679
SPD: -0.07501364839307577
DI: 0.5068107314081718
EOD: -0.005419377001486647
AOD: -0.01870951600905684


### 2.4. Question: Apprentissage de représentation latente fair

Apprendre le pre-processing et evaluer son impact avec les métriques

In [ ]:
print("TODO")

## 3 Post processing

### 3.1 Question: Use the post-processing Reject Option Classification

In [ ]:
print("TODO")

#### 3.1.1 Reuse the first Logistic Regression learn to find the best threshold that maximises its balanced accuracy on the validation dataset

In [ ]:
print("TODO")

#### 3.1.2 Use the RejectOptionClassification  on the validation dataset with the logistic regression predictions. To improve the fairness metrics

In [ ]:
print("TODO")

#### 3.1.3 Do the same while starting from the Logistic Regression learned on the Reweighted dataset

In [ ]:
print("TODO")

#### 3.2 Use the Calibrated Equalised Odds  on the validation dataset with the logistic regression predictions. To improve the fairness metrics

In [ ]:
print("TODO")